<div align="center">

# Hoja de Trabajo 2

**Sofia Garcia 22210**  
**Julio Garcia Salas 22076**

</div>

---

## Ejercicio 1

### 1) Escalabilidad en ABM cuando los atributos **sí** están interconectados (vs. heterogeneidad ortogonal)

Cuando los atributos de los agentes **no** se afectan entre sí (heterogeneidad “ortogonal”), la simulación se vuelve más fácil de escalar: se puede dividir la población en partes, calcular riesgos por separado y casi no se necesita coordinar entre hilos o máquinas. En cambio, si los atributos **sí** están fuertemente conectados (por ejemplo, *ingresos* que cambian mucho el riesgo **según** la *edad*), cada cambio en un atributo toca a los demás y obliga a recalcular más cosas.

En términos sencillos, con ortogonalidad el riesgo suele descomponerse en piezas independientes:

$$
h(t\mid \mathbf{x}) \;\approx\; h_0(t)\,\prod_j \phi_j(x_j)
$$

Con interdependencia fuerte, aparece un **término de interacción** (lo que hace que el efecto de un atributo dependa del otro), así que el riesgo ya no es “por partes”:

$$
h(t\mid \mathbf{x}) \;\propto\; \exp\!\big(\beta_1 x_1 + \beta_2 x_2 + \beta_{12}\,x_1x_2\big)
$$

**Qué cambia en la práctica (y por qué se complica escalar):**
- **Más recomputación:** si cambia ingresos, se altera el riesgo “con edad” y hay que recalcular; ya no se puede reusar tanto lo que estaba cacheado.
- **Más coordinación:** las particiones dejan de ser tan independientes; se necesita comunicar cambios entre bloques (pierde paralelismo “barato”).
- **Más memoria:** se guardan relaciones conjuntas (no solo promedios por atributo), lo que crece más rápido.
- **Calibración más pesada:** no basta con ajustar un promedio por atributo; se ajustan **interacciones** (lo que toma más iteraciones).

**Ideas simples para no perder toda la escalabilidad:**
- Preparar una **población sintética correlacionada** al inicio (ya con las dependencias) y luego **actualizar en lotes** cada $\Delta t$ en vez de cada microcambio.
- **Particionar por comunidades** (grupos con mucha interacción dentro y poca fuera) para reducir la comunicación entre particiones.
- Usar **caché con invalidación selectiva** (solo se invalida lo afectado, no todo el perfil del agente).
- Aproximar riesgos conjuntos con **modelos ligeros** (por ejemplo, una regresión corta) para evitar recomputos costosos en cada paso.

En resumen, cuando hay interdependencia fuerte, se pierde parte del “turbo” de la paralelización fácil. Se gana realismo, pero se paga con más cálculo, más coordinación y más memoria. Aun así, con las estrategias anteriores se conserva buena parte del rendimiento sin sacrificar la idea central del modelo.

---

### 2) Duraciones **fijas** (tiempos de espera) vs. **frecuencias** (riesgo sin memoria) en crónicas: ¿qué se gana y qué se pierde?

Aquí se comparan dos formas de programar “cuándo cambia de estado” un agente:

- **Frecuencias (sin memoria):** el tiempo hasta el evento es exponencial y el “peligro” por unidad de tiempo es constante. Es la versión más simple:
  
  $$
  T \sim \mathrm{Exp}(\lambda),\quad h(t)=\lambda,\quad S(t)=e^{-\lambda t}
  $$

  Ventajas: muy fácil de simular y de calibrar con tasas agregadas. Desventaja: **no recuerda** cuánto tiempo lleva el agente en el estado, lo que puede ser poco realista en crónicas donde el riesgo suele **crecer** con el tiempo.

- **Duraciones fijas (o casi fijas):** se mantiene al agente en un estado por un tiempo típico y luego se cambia. Para evitar que todos cambien exactamente al mismo tiempo, se usa una distribución con poca variabilidad (por ejemplo, Gamma/Erlang):

  $$
  T \sim \mathrm{Gamma}(k,\theta),\quad \mathbb{E}[T]=k\theta,\quad \mathrm{CV}\approx \frac{1}{\sqrt{k}}
  $$

  Ventajas: refleja mejor “fases mínimas” o “latencias” y evita que algunos salten **demasiado** pronto. Desventajas: se debe llevar un pequeño “reloj” por agente y cuidar que no haya sincronías perfectas (se soluciona con un poco de *jitter* o usando Erlang).

**Cuándo conviene cada una (pensando en crónicas):**
- Si se sospecha que el riesgo **aumenta** con el tiempo en estado (muy común en crónicas), conviene una distribución **con memoria**. Una opción práctica es **Weibull** con parámetro de forma $k>1$ (hazard creciente):

  $$
  h(t;k,\lambda) \;=\; k\,\lambda^{k}\,t^{k-1}\quad (k>1 \Rightarrow h(t)\ \text{creciente})
  $$

- Si se está en fase de **prototipo** o solo se tienen datos muy agregados, la exponencial es un buen punto de partida por su simplicidad. Más adelante se puede pasar a Gamma/Weibull para ganar realismo.

- **Duraciones (fijas/casi fijas):** más realismo temporal en crónicas y control de variabilidad; requieren llevar un “reloj” por agente y añadir un poco de aleatoriedad para no sincronizar picos.
- **Frecuencias (exponencial):** muy simple y rápida; útil para comenzar, pero puede **distorsionar** si el riesgo depende de cuánto tiempo se ha permanecido en el estado.

---  